# Pulse: fixture execution and recorded detector evidence

## Goal
Follow actual source code, retain local outputs, then inspect the separate historical detector decision. The fixture produces three accepted windows and three unavailable advisories. It does not fit or qualify a detector.

## Setup
Install the base project and `requirements-notebooks.txt`; run `python scripts/execute_notebooks.py`. Inputs are mathematical fixtures generated by the existing demo and the committed detector report. No dataset download or cloud call occurs.

In [1]:
from pathlib import Path
import hashlib, json, platform
from importlib.metadata import version
from IPython.display import Markdown, display
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'README.md').is_file(), 'Run from the repository root or notebooks directory'
print('Python:', platform.python_version())
print('Notebook runtime:', {p: version(p) for p in ('nbformat', 'nbclient', 'ipykernel')})
def read_evidence(relative):
    raw = (ROOT / relative).read_bytes()
    print(relative, 'SHA-256:', hashlib.sha256(raw).hexdigest())
    return json.loads(raw)


Python: 3.12.14
Notebook runtime: {'nbformat': '5.11.1', 'nbclient': '0.11.0', 'ipykernel': '7.3.0'}


## Steps
### 1. Execute the existing fixture entry point
`run()` calls the source registry, group assignment, validation, descriptive feature calculation, advisory, simulated review and backup. Temporary SQLite/HTML files are removed after the report is captured; use the CLI demo to keep them.

In [2]:
import importlib.util, tempfile
spec = importlib.util.spec_from_file_location('reference_demo', ROOT / 'scripts/run_reference_demo.py')
demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(demo)
print('Demo source SHA-256:', hashlib.sha256((ROOT / 'scripts/run_reference_demo.py').read_bytes()).hexdigest())
print('Core source SHA-256:', hashlib.sha256((ROOT / 'src/pulse/core.py').read_bytes()).hexdigest())
with tempfile.TemporaryDirectory() as output:
    report = demo.run(output)
print(json.dumps({k: v for k, v in report.items() if k != 'results'}, indent=2))

Demo source SHA-256: 1b1d50866360d4871e71d29c31e26edd92c74af2b7bcd91bba9dc0bf00ab7f16
Core source SHA-256: 5fca171ad94acf1f374e52119d84028e2b26c5908d5759a7b4cd28a403305971
{
  "scope": "Generated-fixture software exercise only",
  "site_validated": false,
  "model_trained": false,
  "cloud_deployed": false,
  "counts": {
    "sources": 3,
    "windows": 3,
    "quarantine": 1,
    "evidence": 3,
    "decisions": 3
  }
}


### 2. Read the actual features and advisory boundary
RMS uses m/s². Kurtosis describes the generated signal; neither is a health diagnosis.

In [3]:
rows = []
for result in report['results']:
    rows.append({'asset': result['advisory']['asset_id'], 'features': result['features'],
                 'reason': result['advisory']['reason']})
print(json.dumps(rows, indent=2))

[
  {
    "asset": "fixture-bearing-0",
    "features": {
      "rms_m_s2": 0.7071067811865475,
      "pearson_kurtosis": 1.5000000000000002,
      "feature_version": "rms-pearson-kurtosis-1"
    },
    "reason": "Reference only; unvalidated for customer site. No qualified anomaly model, fault classes or RUL method."
  },
  {
    "asset": "fixture-bearing-1",
    "features": {
      "rms_m_s2": 0.7071067811865475,
      "pearson_kurtosis": 1.5000000000000002,
      "feature_version": "rms-pearson-kurtosis-1"
    },
    "reason": "Reference only; unvalidated for customer site. No qualified anomaly model, fault classes or RUL method."
  },
  {
    "asset": "fixture-bearing-2",
    "features": {
      "rms_m_s2": 0.7071067811865475,
      "pearson_kurtosis": 1.5000000000000002,
      "feature_version": "rms-pearson-kurtosis-1"
    },
    "reason": "Reference only; unvalidated for customer site. No qualified anomaly model, fault classes or RUL method."
  }
]


### 3. Inspect the separate historical detector evaluation
The committed report is a retrospective evaluation of recorded evidence. Loading it today does not rerun a benchmark or reopen a held-out case.

In [4]:
evidence = read_evidence('governance/evidence/detector-performance-consolidated.json')
lines = ['| Candidate | Recorded status | Failed criteria |', '|---|---|---|']
for name, candidate in evidence['candidate_results'].items():
    evaluation = candidate['evaluation']
    failed = ', '.join(k for k,v in evaluation['criteria'].items() if v == 'FAIL') or 'None; criteria inconclusive'
    lines.append(f"| {name} | {evaluation['status']} | {failed} |")
display(Markdown('\n'.join(lines)))

governance/evidence/detector-performance-consolidated.json SHA-256: e29cdb076c6d8110a1301de1d902991f2f86150d7ebd64d476e33327fc6a388e


| Candidate | Recorded status | Failed criteria |
|---|---|---|
| frozen-3x-rms | INCONCLUSIVE | None; criteria inconclusive |
| sca-paired-development | FAIL | DET-PERF-004, DET-PERF-006 |

## Checks
Check the fixture boundary and preserve historical non-release. These are software/evidence consistency checks, not new model acceptance.

In [5]:
assert len(report['results']) == 3
assert report['model_trained'] is False and report['site_validated'] is False
assert evidence['production_release_authorized'] is False
assert evidence['candidate_results']['sca-paired-development']['evaluation']['status'] == 'FAIL'
print('PASS: fixture scope and recorded failed detector disposition preserved')

PASS: fixture scope and recorded failed detector disposition preserved


## Next Steps
The fixture shows the implemented pipeline and its abstention behavior. The paired candidate remains failed; frozen RMS remains inconclusive. Review [results](../docs/results.md), the linked measurement records and current detector backlog. Raw vibration EDA and new model comparisons remain separate engineering work; no MLflow run is claimed by this notebook.